# 01 — Exploratory Data Analysis

Loads the raw listings dataset, applies the cleaning pipeline (`src/preprocessing/cleaning.py`), and reproduces the headline EDA figures from `reports/technical_report.md` §2.

This notebook is illustrative — it is **not** used to train models. Training happens via `train.py` / `benchmark.py`, which call the same `src/` modules directly.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.preprocessing.cleaning import clean_pipeline
from src.preprocessing.validation import validate_dataset
from src.utils.config import resolve_path
from src.visualization.plots import save_figure, set_publication_style

set_publication_style()

## Load and clean

In [ ]:
df = clean_pipeline(resolve_path("raw_listings"))
print(f"Cleaned shape: {df.shape}")
validate_dataset(df)

## Revenue distribution — raw vs. log1p

Raw TTM revenue is heavily right-skewed (skewness ≈ 4.2); after a `log1p` transform, skewness drops to ≈ 0.31 — this is why every model in this project trains on `log1p(ttm_revenue)`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["ttm_revenue"], bins=60, color="steelblue", edgecolor="white")
axes[0].set_title("TTM Revenue — raw scale")
axes[0].set_xlabel("Revenue (USD)")
axes[1].hist(np.log1p(df["ttm_revenue"]), bins=60, color="darkorange", edgecolor="white")
axes[1].set_title("TTM Revenue — log1p scale")
axes[1].set_xlabel("log(1 + Revenue)")
save_figure(fig, resolve_path("figure_dir") / "figure1_revenue_distribution.png")

## Median revenue by country (top 12)

In [ ]:
top_countries = df.groupby("country")["ttm_revenue"].median().sort_values(ascending=False).head(12)
fig, ax = plt.subplots(figsize=(10, 5))
top_countries.plot(kind="bar", ax=ax, color="teal", edgecolor="white")
ax.set_title("Median TTM Revenue by Country (top 12)")
ax.set_ylabel("Median Revenue (USD)")
plt.xticks(rotation=45, ha="right")
save_figure(fig, resolve_path("figure_dir") / "figure2_revenue_by_country.png")

## Correlation heatmap of key numeric features

In [ ]:
import seaborn as sns

corr_cols = ["ttm_revenue", "ttm_occupancy", "ttm_avg_rate", "guests", "bedrooms",
             "beds", "baths", "num_reviews", "rating_overall", "cleaning_fee", "min_nights"]
corr_df = df[corr_cols].dropna()
mask = np.triu(np.ones_like(corr_df.corr(), dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_df.corr(), mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title("Correlation Heatmap — Key Features")
save_figure(fig, resolve_path("figure_dir") / "figure4_correlation_heatmap.png")

## Save the cleaned dataset

This is the file `train.py`, `evaluate.py`, `benchmark.py` and `predict.py` all read via `configs/paths.yaml: processed_dataset`.

In [ ]:
from src.utils.io import read_table, write_table
out_path = resolve_path("processed_dataset")
out_path.parent.mkdir(parents=True, exist_ok=True)
write_table(df, out_path)
print(f"Saved cleaned dataset -> {out_path}")